**1. Aim**
To create a DataFrame using PySpark and perform data cleaning operations such as handling missing values, removing duplicate records, correcting data types, and performing feature processing to prepare the dataset for further analysis.








**2. Related Theory**
PySpark is the Python API for Apache Spark, an open-source distributed computing framework used to process large datasets efficiently. It allows users to perform data processing and analysis using Python.
A DataFrame in PySpark is a distributed collection of data organized into named columns, similar to a table in a relational database. It supports operations such as filtering, selecting, grouping, and transforming data.
Data cleaning is the process of identifying and correcting problems in a dataset. These problems may include missing values, duplicate records, incorrect data types, and inconsistent data. Missing values can be handled using methods such as removing records with dropna() or filling missing values with fillna(). Duplicate records can be removed using dropDuplicates().
Feature processing prepares data for analysis and machine learning. It includes converting data types, encoding categorical values, and creating or transforming features. For example, a categorical attribute such as gender can be converted into a numerical representation using indexing or one-hot encoding.
In this experiment, PySpark is used to create a DataFrame, inspect the data, clean missing and duplicate values, and process selected features to prepare a cleaner dataset for further analysis.




In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

spark = SparkSession.builder \
    .appName("DataCleaningAndFeatureProcessing") \
    .getOrCreate()

print("Spark session created successfully")

Spark session created successfully


Create a sample DataFrame

In [ ]:
data = [
    (1, "Amit", 20, "Male", 85.0),
    (2, "Priya", 21, "Female", 90.0),
    (3, "Rahul", None, "Male", 78.0),
    (4, "Sneha", 22, "Female", None),
    (5, "Amit", 20, "Male", 85.0),
    (6, "Rohan", 23, None, 88.0)
]

columns = ["Student_ID", "Name", "Age", "Gender", "Marks"]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

+----------+-----+----+------+-----+
|Student_ID| Name| Age|Gender|Marks|
+----------+-----+----+------+-----+
|         1| Amit|  20|  Male| 85.0|
|         2|Priya|  21|Female| 90.0|
|         3|Rahul|NULL|  Male| 78.0|
|         4|Sneha|  22|Female| NULL|
|         5| Amit|  20|  Male| 85.0|
|         6|Rohan|  23|  NULL| 88.0|
+----------+-----+----+------+-----+

root
 |-- Student_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Marks: double (nullable = true)



Identify missing values

In [ ]:
from pyspark.sql.functions import sum, when

df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+----------+----+---+------+-----+
|Student_ID|Name|Age|Gender|Marks|
+----------+----+---+------+-----+
|         0|   0|  1|     1|    1|
+----------+----+---+------+-----+



Handle missing values

In [ ]:
from pyspark.sql.functions import avg

average_marks = df.select(avg("Marks")).first()[0]

df_clean = df.fillna({
    "Age": 0,
    "Gender": "Unknown",
    "Marks": average_marks
})

df_clean.show()

+----------+-----+---+-------+-----+
|Student_ID| Name|Age| Gender|Marks|
+----------+-----+---+-------+-----+
|         1| Amit| 20|   Male| 85.0|
|         2|Priya| 21| Female| 90.0|
|         3|Rahul|  0|   Male| 78.0|
|         4|Sneha| 22| Female| 85.2|
|         5| Amit| 20|   Male| 85.0|
|         6|Rohan| 23|Unknown| 88.0|
+----------+-----+---+-------+-----+



Remove duplicate records

In [ ]:
df_clean = df_clean.dropDuplicates()

df_clean.show()

+----------+-----+---+-------+-----+
|Student_ID| Name|Age| Gender|Marks|
+----------+-----+---+-------+-----+
|         3|Rahul|  0|   Male| 78.0|
|         2|Priya| 21| Female| 90.0|
|         1| Amit| 20|   Male| 85.0|
|         5| Amit| 20|   Male| 85.0|
|         4|Sneha| 22| Female| 85.2|
|         6|Rohan| 23|Unknown| 88.0|
+----------+-----+---+-------+-----+



Correct data types

In [ ]:
df_clean = df_clean.withColumn(
    "Age", col("Age").cast("integer")
).withColumn(
    "Marks", col("Marks").cast("double")
)

df_clean.printSchema()

root
 |-- Student_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Gender: string (nullable = false)
 |-- Marks: double (nullable = false)



Encode categorical features

In [ ]:
indexer = StringIndexer(
    inputCol="Gender",
    outputCol="Gender_Index",
    handleInvalid="keep"
)

indexer_model = indexer.fit(df_clean)
df_indexed = indexer_model.transform(df_clean)

df_indexed.select(
    "Name", "Gender", "Gender_Index"
).show()

+-----+-------+------------+
| Name| Gender|Gender_Index|
+-----+-------+------------+
|Rahul|   Male|         0.0|
|Priya| Female|         1.0|
| Amit|   Male|         0.0|
| Amit|   Male|         0.0|
|Sneha| Female|         1.0|
|Rohan|Unknown|         2.0|
+-----+-------+------------+



Convert the indexed feature into one-hot encoding

In [ ]:
encoder = OneHotEncoder(
    inputCol="Gender_Index",
    outputCol="Gender_Vector"
)

encoder_model = encoder.fit(df_indexed)
df_encoded = encoder_model.transform(df_indexed)

df_encoded.select(
    "Name", "Gender", "Gender_Index", "Gender_Vector"
).show(truncate=False)

+-----+-------+------------+-------------+
|Name |Gender |Gender_Index|Gender_Vector|
+-----+-------+------------+-------------+
|Rahul|Male   |0.0         |(3,[0],[1.0])|
|Priya|Female |1.0         |(3,[1],[1.0])|
|Amit |Male   |0.0         |(3,[0],[1.0])|
|Amit |Male   |0.0         |(3,[0],[1.0])|
|Sneha|Female |1.0         |(3,[1],[1.0])|
|Rohan|Unknown|2.0         |(3,[2],[1.0])|
+-----+-------+------------+-------------+



Assemble features into a vector

In [ ]:
assembler = VectorAssembler(
    inputCols=["Age", "Marks", "Gender_Index"],
    outputCol="Features"
)

df_final = assembler.transform(df_encoded)

df_final.select(
    "Student_ID", "Name", "Features"
).show(truncate=False)

+----------+-----+---------------+
|Student_ID|Name |Features       |
+----------+-----+---------------+
|3         |Rahul|[0.0,78.0,0.0] |
|2         |Priya|[21.0,90.0,1.0]|
|1         |Amit |[20.0,85.0,0.0]|
|5         |Amit |[20.0,85.0,0.0]|
|4         |Sneha|[22.0,85.2,1.0]|
|6         |Rohan|[23.0,88.0,2.0]|
+----------+-----+---------------+



Display the final DataFrame

In [ ]:
df_final.show(truncate=False)
df_final.printSchema()

+----------+-----+---+-------+-----+------------+-------------+---------------+
|Student_ID|Name |Age|Gender |Marks|Gender_Index|Gender_Vector|Features       |
+----------+-----+---+-------+-----+------------+-------------+---------------+
|3         |Rahul|0  |Male   |78.0 |0.0         |(3,[0],[1.0])|[0.0,78.0,0.0] |
|2         |Priya|21 |Female |90.0 |1.0         |(3,[1],[1.0])|[21.0,90.0,1.0]|
|1         |Amit |20 |Male   |85.0 |0.0         |(3,[0],[1.0])|[20.0,85.0,0.0]|
|5         |Amit |20 |Male   |85.0 |0.0         |(3,[0],[1.0])|[20.0,85.0,0.0]|
|4         |Sneha|22 |Female |85.2 |1.0         |(3,[1],[1.0])|[22.0,85.2,1.0]|
|6         |Rohan|23 |Unknown|88.0 |2.0         |(3,[2],[1.0])|[23.0,88.0,2.0]|
+----------+-----+---+-------+-----+------------+-------------+---------------+

root
 |-- Student_ID: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = false)
 |-- Gender: string (nullable = false)
 |-- Marks: double (nullable = false)
 |--

Conclusion

The experiment was successfully performed using PySpark in Google Colab. A DataFrame was created, and data cleaning operations such as identifying and handling missing values, removing duplicate records, and correcting data types were implemented. Feature processing was performed by encoding categorical attributes and assembling selected columns into a feature vector. Thus, the experiment demonstrated how PySpark can be used to prepare structured data for further analysis and machine learning applications.
